In [88]:
import pandas as pd
import re

# 파일 경로 설정 (같은 폴더 내에 위치할 것)
file_zootopia = 'Megabox_Zootopia2_Seoul_1129.csv'
file_ranking = 'megaboxCrawling.csv'

# 크롤링된 데이터 로드
df_zoo = pd.read_csv(file_zootopia)
df_rank = pd.read_csv(file_ranking)

# 예매율 숫자만 추출 (예: '예매율 54.2%' -> 54.2)
def clean_booking_rate(val):
    if isinstance(val, str):
        numbers = re.findall(r"[-+]?\d*\.\d+|\d+", val)
        return float(numbers[0]) if numbers else 0.0
    return val

# 개봉일 날짜 형식 변환
def clean_date(val):
    if isinstance(val, str):
        return val.replace('개봉일 ', '').strip()
    return val

df_rank['booking_rate_num'] = df_rank['booking_rate'].apply(clean_booking_rate)
df_rank['release_date_dt'] = pd.to_datetime(df_rank['release_date'].apply(clean_date), errors='coerce')

print("데이터 로드 및 전처리 완료")

데이터 로드 및 전처리 완료


In [89]:
# 예매된 좌석 수 계산
df_zoo['예매좌석'] = df_zoo['전체좌석'] - df_zoo['잔여좌석']

# 극장별로 그룹화하여 합계 계산
theater_stats = df_zoo.groupby('극장')[['전체좌석', '예매좌석']].sum()

# 극장별 평균 예매율 계산
theater_stats['예매율(%)'] = (theater_stats['예매좌석'] / theater_stats['전체좌석']) * 100

# 예매율이 높은 순서대로 정렬
theater_stats_sorted = theater_stats.sort_values(by='예매율(%)', ascending=False)

print("\n=== [분석 1] 주토피아2 서울 극장별 예매율 ===")
theater_stats_sorted[['예매좌석', '전체좌석', '예매율(%)']]


=== [분석 1] 주토피아2 서울 극장별 예매율 ===


,예매좌석,전체좌석,예매율(%)
극장,,,
강남,922,2222,41.494149
코엑스,3717,10070,36.911619
센트럴,1065,2973,35.822402
홍대,682,2266,30.097087
동대문,569,2026,28.084896
구의이스트폴,777,3189,24.365005
더부티크목동현대백화점,1042,4368,23.855311
화곡,439,1864,23.551502
이수,738,4173,17.685119


In [90]:
# --- [수정된 분석 2-1] 유의미한 예매율을 가진 최신 영화 TOP 5 ---

# 1. 예매율이 0보다 큰 영화만 필터링 (미래 개봉작이나 데이터 없는 영화 제외)
active_movies = df_rank[df_rank['booking_rate_num'] > 0]

# 2. 개봉일 기준 내림차순 정렬 (최신순)
latest_active_movies = active_movies.sort_values(by='release_date_dt', ascending=False)

# 3. 결과 출력 (상위 5개만)
target_columns = ['title', 'booking_rate', 'release_date', 'like']  # index = ['제목', '예매율(%)', '개봉일', '좋아요 수']
print("\n=== 개봉 예정 영화 중 예매율 비교 ===")
latest_active_movies[target_columns].head(15)


=== 개봉 예정 영화 중 예매율 비교 ===


,title,booking_rate,release_date,like
23,KESSOKUBAND LIVE IN KOREA 「FROM SHIMOKITAZAWA」,예매율 0.3%,개봉일 2025.12.06,81
37,[AGF2025] 극장판 진격의 거인 완결편 더 라스트 어택,예매율 0.1%,개봉일 2025.12.05,29
20,[AGF2025] 극장판 총집편 걸즈 밴드 크라이 청춘광주곡,예매율 0.3%,개봉일 2025.12.05,66
27,[AGF2025] 우마무스메 프리티 더비 새로운 시대의 문,예매율 0.2%,개봉일 2025.12.05,36
18,[AGF2025] 영화 러브 라이브! 니지가사키 학원 스쿨 아이돌 동호회 완결편 제2장,예매율 0.4%,개봉일 2025.12.05,58
28,[AGF2025] 극장판 프로젝트 세카이 부서진 세카이와 전해지지 않는 미쿠의 노래,예매율 0.1%,개봉일 2025.12.05,27
30,[AGF2025] RE:제로부터 시작하는 이세계 생활 4기 EP 1,예매율 0.1%,개봉일 2025.12.05,25
9,에이티즈 브이알 콘서트 : 라이트 더 웨이,예매율 1.8%,개봉일 2025.12.05,438
26,[AGF2025] 페이트 스트레인지 페이크 EP 1-4,예매율 0.2%,개봉일 2025.12.05,32
42,"[AGF2025] 극장판 뱅드림! 잇츠 마이고!!!!! 전편: 봄의 양지, 방황하는...",예매율 0.1%,개봉일 2025.12.05,17


In [91]:
# 개봉일 기준 내림차순 정렬
active_movies = df_rank[df_rank['booking_rate_num'] > 0]
latest_movies = active_movies.sort_values(by='release_date', ascending=False).head(30)

print("\n=== [분석 2] 최근 개봉(및 예정) 영화 리스트 ===")
latest_movies[['title', 'release_date', 'booking_rate']]


=== [분석 2] 최근 개봉(및 예정) 영화 리스트 ===


,title,release_date,booking_rate
23,KESSOKUBAND LIVE IN KOREA 「FROM SHIMOKITAZAWA」,개봉일 2025.12.06,예매율 0.3%
37,[AGF2025] 극장판 진격의 거인 완결편 더 라스트 어택,개봉일 2025.12.05,예매율 0.1%
20,[AGF2025] 극장판 총집편 걸즈 밴드 크라이 청춘광주곡,개봉일 2025.12.05,예매율 0.3%
27,[AGF2025] 우마무스메 프리티 더비 새로운 시대의 문,개봉일 2025.12.05,예매율 0.2%
18,[AGF2025] 영화 러브 라이브! 니지가사키 학원 스쿨 아이돌 동호회 완결편 제2장,개봉일 2025.12.05,예매율 0.4%
28,[AGF2025] 극장판 프로젝트 세카이 부서진 세카이와 전해지지 않는 미쿠의 노래,개봉일 2025.12.05,예매율 0.1%
30,[AGF2025] RE:제로부터 시작하는 이세계 생활 4기 EP 1,개봉일 2025.12.05,예매율 0.1%
9,에이티즈 브이알 콘서트 : 라이트 더 웨이,개봉일 2025.12.05,예매율 1.8%
26,[AGF2025] 페이트 스트레인지 페이크 EP 1-4,개봉일 2025.12.05,예매율 0.2%
42,"[AGF2025] 극장판 뱅드림! 잇츠 마이고!!!!! 전편: 봄의 양지, 방황하는...",개봉일 2025.12.05,예매율 0.1%


In [92]:
# 랭킹 컬럼에서 숫자만 추출하여 정렬 기준 만들기 (예: '1위' -> 1)
df_rank['rank_num'] = df_rank['rank'].str.extract('(\d+)').astype(int)

# 랭킹순 정렬
popular_movies = df_rank.sort_values(by='rank_num')

print("\n=== [분석 3] 현재 인기 영화 TOP 10 (예매율 vs 좋아요 비교) ===")
# 주요 컬럼만 출력
comparison_df = popular_movies[['rank', 'title', 'booking_rate', 'like']].head(10)
comparison_df


=== [분석 3] 현재 인기 영화 TOP 10 (예매율 vs 좋아요 비교) ===


<>:2: SyntaxWarning: invalid escape sequence '\d'
<>:2: SyntaxWarning: invalid escape sequence '\d'
C:\Users\user\AppData\Local\Temp\ipykernel_18312\3817520476.py:2: SyntaxWarning: invalid escape sequence '\d'
  df_rank['rank_num'] = df_rank['rank'].str.extract('(\d+)').astype(int)


,rank,title,booking_rate,like
0,1위,주토피아 2,예매율 54.2%,1.7k
1,2위,위키드: 포 굿,예매율 7.1%,1.6k
2,3위,극장판 주술회전: 시부야사변 X 사멸회유,예매율 5.5%,1.2k
3,4위,정보원,예매율 5.5%,120
4,5위,윗집 사람들,예매율 4.4%,211
5,6위,극장판 체인소 맨: 레제편,예매율 2.9%,5.4k
6,7위,반지의 제왕: 두 개의 탑,예매율 2.8%,964
7,8위,나우 유 씨 미 3,예매율 2.7%,864
8,9위,국보,예매율 1.9%,930
9,10위,에이티즈 브이알 콘서트 : 라이트 더 웨이,예매율 1.8%,438
